# NIR: Замена этапа оптимальной интерполяции Максимова на нейросеть с оценкой неопределённости

**Пайплайны P1–P6:**
| # | Этап 1 | Этап 3 |
|---|---|---|
| P1 | Бикубическая интерполяция | Усреднение |
| P2 | Максимов (Wiener-фильтр + аналит. D_ε) | Взвешенное (формула 92) |
| P3 | EDSR | Усреднение |
| P4 | EDSR + TTA-ансамбль | Взвешенное по дисперсии ансамбля |
| P5 | EDSR + MC Dropout | Взвешенное по дисперсии MCD |
| P6 | EDSR + гетероскедастич. прокси | Взвешенное по предсказанной σ² |

**Этап 2** (геометрическое согласование ECC-пирамида) — общий для всех вариантов.

In [83]:
# ─── 0. IMPORTS & CONFIG ────────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')
import time, math
import numpy as np
import cv2
import matplotlib; import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats, ndimage
from skimage.metrics import structural_similarity as ssim_func
import torch
import torch.nn as nn
from pathlib import Path

ROOT        = Path('.')
MODEL_PATH  = ROOT / 'models' / 'edsr_finetuned.pth'
HR_DIR      = ROOT / 'dataset' / 'DIV2K' / 'DIV2K_valid_HR'
# ── FIX #1: реальное тестовое изображение (номерной знак 100×200) ─────────
TEST_IMG_PATH = ROOT / 'dataset' / 'test' / 'Test.png'

# ── Experiment knobs ───────────────────────────────────────────────────────
SCALE       = 4      # EDSR upsampling factor (fixed ×4)
SCALE_DOWN  = 6      # LR generation downsampling factor
NOISE_RATIO = 0.15   # D_v / D_x
BLUR_D      = 0.15   # PSF width d (fraction of T²)
N_FRAMES    = 12     # LR frames per scene
MCD_K       = 10     # MC Dropout forward passes  ← NOT overwritten globally anymore
TTA_N       = 8      # TTA ensemble size (FIX #2: 8 аугментаций вместо 4)
RHO         = 0.90   # signal autocorrelation ρ

MODES  = ['P1','P2','P3','P4','P5','P6']
LABELS = ['P1: Bicubic+avg','P2: Maximov','P3: EDSR+avg',
          'P4: EDSR+TTA8','P5: EDSR+MCD','P6: EDSR+Heterosc.']
COLORS = ['#6b7280','#2563eb','#16a34a','#f59e0b','#dc2626','#7c3aed']

matplotlib.rcParams.update({'figure.dpi':110,'font.size':11,
                             'axes.grid':True,'grid.alpha':0.3})
torch.manual_seed(0); np.random.seed(0)
print(f'PyTorch {torch.__version__} | OpenCV {cv2.__version__}')

# ── Fast mode ────────────────────────────────────────────────────────────────
# FIX #6: SWEEP_MCD_K используется только как аргумент, не перезаписывает MCD_K
FAST_MODE    = True
SWEEP_FRAMES = 4  if FAST_MODE else N_FRAMES
SWEEP_MCD_K  = 3  if FAST_MODE else MCD_K   # передаётся в функцию, не в глобал


PyTorch 2.11.0+cpu | OpenCV 4.13.0


## 1. Metrics
PSNR, SSIM, normalized error D_eps/D_x, Spearman correlation, AUSE, reliability diagram.

In [84]:
# ─── 1. METRICS ─────────────────────────────────────────────────────────────

def psnr(gt, pred):
    """Peak Signal-to-Noise Ratio in dB. Higher is better."""
    mse = np.mean((gt.astype(float) - np.clip(pred,0,255).astype(float))**2)
    return float('inf') if mse == 0 else 10 * np.log10(255**2 / mse)

def ssim_score(gt, pred):
    """Structural Similarity. Higher is better."""
    return ssim_func(gt.astype(np.uint8),
                     np.clip(pred,0,255).astype(np.uint8), data_range=255)

def norm_error(gt, pred):
    """Normalised reconstruction error D_eps/D_x (formula 64). Lower is better."""
    g = gt.astype(float) / 255.0
    p = np.clip(pred,0,255).astype(float) / 255.0
    return np.mean((g - p)**2) / (np.var(g) + 1e-8)

def spearman_corr(err_map, unc_map):
    """Spearman rank correlation between actual error and uncertainty map."""
    r, _ = stats.spearmanr(err_map.ravel(), unc_map.ravel())
    return float(r)

def ause(err_map, unc_map, n_bins=20):
    """Area Under Sparsification Error. Lower is better.
    Remove pixels with highest uncertainty step-by-step; compute remaining MSE."""
    flat_err = err_map.ravel()
    flat_unc = unc_map.ravel()
    order = np.argsort(flat_unc)[::-1]   # highest uncertainty first
    fracs = np.linspace(0, 0.9, n_bins)
    errors = []
    for f in fracs:
        keep = order[int(f * len(order)):]
        errors.append(np.mean(flat_err[keep]**2) if len(keep) else 0)
    return float(np.trapezoid(errors, fracs))

def reliability_diagram(gt, pred, unc_map, n_bins=10):
    """Compute binned (predicted_unc, actual_error) pairs for reliability plot."""
    err = np.abs(gt.astype(float) - np.clip(pred,0,255).astype(float)) / 255.0
    unc = (unc_map - unc_map.min()) / (unc_map.max() - unc_map.min() + 1e-8)
    bins = np.linspace(0, 1, n_bins + 1)
    xs, ys = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (unc >= lo) & (unc < hi)
        if mask.sum() > 10:
            xs.append(unc[mask].mean())
            ys.append(err[mask].mean())
    return np.array(xs), np.array(ys)

## 2. Degradation model (Maximov)
  with Gaussian PSF, decimation, and additive noise.

In [85]:
# ─── 2. DEGRADATION MODEL (Maximov's observation model) ─────────────────────
# y[k] = (h * x)[kT + shift] + v[k],   v ~ N(0, D_v)

def degrade(hr, scale, blur_d, noise_ratio, shift_xy):
    """
    Generate one LR frame from a HR image.
    hr         : float32 [0..255], shape (H, W)
    scale      : downsampling factor L
    blur_d     : PSF width d; sigma = sqrt(d) * scale  (in HR pixels)
    noise_ratio: D_v / D_x
    shift_xy   : (dx, dy) subpixel shift in LR pixels
    Returns    : float32 [0..255], shape (H//L, W//L)
    """
    sigma = math.sqrt(max(blur_d, 0)) * scale
    if sigma > 0:
        ks = max(3, int(6 * sigma + 1) | 1)
        img = cv2.GaussianBlur(hr, (ks, ks), sigma)
    else:
        img = hr.copy()

    dx, dy = shift_xy
    M = np.float32([[1, 0, dx], [0, 1, dy]])
    img = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)

    h, w = img.shape[:2]
    lr = cv2.resize(img, (w // scale, h // scale), interpolation=cv2.INTER_AREA)

    # FIX BUG2-v2: IQR-based D_x — robust for bimodal images (license plates).
    # For mostly-white LR (q25≈q75≈1), IQR≈0 → floor → low noise; 
    # for natural images the IQR stays large → noise unchanged. 
    # NOISE_RATIO в ячейке 0 НЕ меняется.
    lr_f = lr.astype(float) / 255.0
    q25, q75 = np.percentile(lr_f, [25, 75])
    D_x = max(((q75 - q25) / 1.35) ** 2, 0.01)   # IQR → σ (normal-consistent)
    std = math.sqrt(max(noise_ratio * D_x, 0)) * 255.0
    noise = np.random.normal(0, std, lr.shape) if std > 0 else 0
    return np.clip(lr.astype(float) + noise, 0, 255).astype(np.float32)


# FIX #5: возвращаем сдвиги вместе с кадрами — они нужны для D_eps
def generate_lr_series(hr, n_frames, scale, blur_d, noise_ratio, max_shift=1.0):
    """Generate LR frames with random subpixel shifts.
    Returns: (list_of_lr_frames, list_of_shifts)  ← ИЗМЕНЕНО
    """
    shifts = [(np.random.uniform(-max_shift, max_shift),
               np.random.uniform(-max_shift, max_shift))
              for _ in range(n_frames)]
    frames = [degrade(hr, scale, blur_d, noise_ratio, s) for s in shifts]
    return frames, shifts


## 3. Load test data
Load a 256×256 grayscale crop from DIV2K validation set and generate LR series.

In [86]:
# ─── 3. LOAD TEST IMAGE & VISUALISE DEGRADATION ─────────────────────────────
# FIX #1: используем dataset/test/Test.png (номерной знак 100×200)
# DownScale x6: ~17×33, UpScale x4 (EDSR) → ~67×133

def load_gray(path):
    """Load grayscale image as float32."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f'Image not found: {path}')
    return img.astype(np.float32)

def load_gray_crop(path, crop=256):
    """Load grayscale image and take a centre crop."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE).astype(np.float32)
    h, w = img.shape
    y, x = (h - crop) // 2, (w - crop) // 2
    return img[y:y+crop, x:x+crop]

# ── Основной тестовый образ: Test.png ────────────────────────────────────────
HR = load_gray(TEST_IMG_PATH)   # 100×200 — номерной знак
np.random.seed(42)
LR_SERIES, LR_SHIFTS = generate_lr_series(HR, N_FRAMES, SCALE_DOWN, BLUR_D, NOISE_RATIO)

print(f'HR (Test.png): {HR.shape}  →  LR: {LR_SERIES[0].shape}  (downscale x{SCALE_DOWN}, EDSR upscale x{SCALE})')
# HR reference at EDSR output resolution (H*SCALE/SCALE_DOWN × W*SCALE/SCALE_DOWN)
lr0 = LR_SERIES[0]
HR_REF = cv2.resize(HR, (lr0.shape[1]*SCALE, lr0.shape[0]*SCALE), interpolation=cv2.INTER_LANCZOS4)
print(f'LR: {LR_SERIES[0].shape}  →  EDSR out / HR_REF: {HR_REF.shape}')

fig, axes = plt.subplots(2, 4, figsize=(14, 5))
axes[0,0].imshow(HR, cmap='gray'); axes[0,0].set_title(f'HR Original\n{HR.shape[1]}×{HR.shape[0]}', fontsize=9)
for i in range(3):
    axes[0, i+1].imshow(LR_SERIES[i], cmap='gray', vmin=0, vmax=255)
    axes[0, i+1].set_title(f'LR кадр {i+1}\n{LR_SERIES[i].shape[1]}×{LR_SERIES[i].shape[0]}\nshift={LR_SHIFTS[i][0]:.2f},{LR_SHIFTS[i][1]:.2f}', fontsize=8)

# Покажем также bicycle-upscale и насколько он отличается от GT
lr0_up = cv2.resize(LR_SERIES[0], (LR_SERIES[0].shape[1]*SCALE, LR_SERIES[0].shape[0]*SCALE), interpolation=cv2.INTER_CUBIC)
axes[1,0].imshow(HR, cmap='gray'); axes[1,0].set_title('GT HR', fontsize=9)
axes[1,1].imshow(LR_SERIES[0], cmap='gray', vmin=0, vmax=255); axes[1,1].set_title('LR кадр 1 (нечитаем)', fontsize=9)
axes[1,2].imshow(lr0_up, cmap='gray', vmin=0, vmax=255); axes[1,2].set_title(f'LR×{SCALE} bicubic\n(для сравнения)', fontsize=9)
err = np.abs(HR_REF - lr0_up)
axes[1,3].imshow(err, cmap='hot'); axes[1,3].set_title(f'|HR_REF - bicubic|\nPSNR={10*np.log10(255**2/(np.mean(err**2)+1e-8)):.1f}dB', fontsize=9)

for ax in axes.ravel(): ax.axis('off')
plt.suptitle(f'Test.png: HR ({HR.shape[1]}×{HR.shape[0]}) → LR×{SCALE_DOWN} ({LR_SERIES[0].shape[1]}×{LR_SERIES[0].shape[0]}) → EDSR×{SCALE} ({HR_REF.shape[1]}×{HR_REF.shape[0]})', fontsize=11)
plt.tight_layout(); plt.show()

# ── DIV2K для sweep-экспериментов (нужен crop ≥ 128 для стабильной регистрации) ──
HR_SMALL = load_gray_crop(HR_DIR / '0801.png', crop=128)
HR_SMALL_REF = cv2.resize(HR_SMALL, (HR_SMALL.shape[1]//SCALE_DOWN*SCALE, HR_SMALL.shape[0]//SCALE_DOWN*SCALE), interpolation=cv2.INTER_LANCZOS4)
print(f'HR_SMALL (для sweep): {HR_SMALL.shape}  →  HR_SMALL_REF: {HR_SMALL_REF.shape}')


HR (Test.png): (100, 200)  →  LR: (16, 33)  (downscale x6, EDSR upscale x4)
LR: (16, 33)  →  EDSR out / HR_REF: (64, 132)
HR_SMALL (для sweep): (128, 128)  →  HR_SMALL_REF: (84, 84)


## 4. Stage 1a — Maximov optimal filter
Wiener deconvolution in frequency domain. Analytical per-pixel error variance D_eps.

In [87]:
# ─── 4. STAGE 1a: MAXIMOV — Wiener filter + analytical D_eps ────────────────

def wiener_deconvolve(lr_up, sigma, noise_ratio):
    """
    Wiener deconvolution in frequency domain.
    G(f) = H*(f) / (|H(f)|^2 + lambda),   lambda = noise_ratio
    """
    img = lr_up.astype(float) / 255.0
    H_rows, H_cols = img.shape
    fy = np.fft.fftfreq(H_rows)[:, None]
    fx = np.fft.fftfreq(H_cols)[None, :]
    H_psf = np.exp(-2 * np.pi**2 * sigma**2 * (fy**2 + fx**2))
    lam   = max(noise_ratio, 1e-4)
    G     = H_psf / (H_psf**2 + lam)
    X_hat = np.real(np.fft.ifft2(np.fft.fft2(img) * G))
    return np.clip(X_hat, 0, 1) * 255.0


# FIX #5: D_eps теперь вычисляется из реального сдвига кадра, а не периодического grid-offset
def maximov_deps_map_from_shift(shape, shift_xy, noise_ratio, rho=RHO):
    """
    Per-frame error variance map D_eps based on actual subpixel shift.
    D_eps(delta) = D_x*(1 - rho^(2*delta)) + noise_ratio
    где delta — нормированное расстояние субпиксельного сдвига (0..1).

    Для равномерного сдвига (dx, dy) в LR-пикселях:
    delta = |frac(dx), frac(dy)| / (sqrt(2)/2), обрезанное в [0, 1].
    Возвращает равномерную карту (одно значение на весь кадр).
    """
    dx, dy = shift_xy
    # дробная часть сдвига — субпиксельное смещение от ближайшего узла решётки
    frac_x = dx - math.floor(dx)   # в [0, 1)
    frac_y = dy - math.floor(dy)
    # расстояние до ближайшего узла LR-решётки
    dist_x = min(frac_x, 1.0 - frac_x)
    dist_y = min(frac_y, 1.0 - frac_y)
    delta  = math.hypot(dist_x, dist_y) / (math.sqrt(2) / 2.0)
    delta  = min(delta, 1.0)
    D_eps  = (1.0 - rho ** (2.0 * delta)) + noise_ratio
    return np.full(shape, max(D_eps, 1e-6), dtype=np.float32)


def maximov_deps_map_grid(shape, scale, noise_ratio, rho=RHO):
    """
    Старая реализация (для совместимости): D_eps по периодическому grid-offset.
    Используется когда реальный сдвиг неизвестен.
    """
    H, W = shape
    dr = (np.arange(H) % scale) / scale
    dc = (np.arange(W) % scale) / scale
    dr, dc = np.meshgrid(dr, dc, indexing='ij')
    delta = np.clip(np.hypot(dr, dc) / (math.sqrt(2) / 2), 0, 1)
    D_eps = (1 - rho**(2 * delta)) + noise_ratio
    return np.maximum(D_eps, 1e-6).astype(np.float32)


def maximov_stage1(lr, scale, blur_d, noise_ratio, shift_xy=None):
    """Maximov optimal filter. Returns (hr_restored, D_eps_map).
    FIX #5: если передан shift_xy — используется точная D_eps для данного кадра.
    """
    h, w = lr.shape
    lr_up = cv2.resize(lr, (w * scale, h * scale), interpolation=cv2.INTER_CUBIC)
    sigma = math.sqrt(max(blur_d, 0.05)) * scale
    restored = wiener_deconvolve(lr_up, sigma, noise_ratio)
    if shift_xy is not None:
        D_eps = maximov_deps_map_from_shift(restored.shape, shift_xy, noise_ratio)
    else:
        D_eps = maximov_deps_map_grid(restored.shape, scale, noise_ratio)
    return restored.astype(np.float32), D_eps


## 5. EDSR neural network
EDSR-baseline: 16 ResBlocks, 64 features, x4 upsampling, RGB in/out. Load pretrained weights.

In [88]:
# ─── 5. EDSR MODEL ───────────────────────────────────────────────────────────
# Architecture from saved weights:
# RGB in/out, 16 ResBlocks, 64 features, x4 (two PixelShuffle x2 stages)

class ResBlock(nn.Module):
    def __init__(self, n_feats=64, res_scale=0.1):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(n_feats, n_feats, 3, 1, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(n_feats, n_feats, 3, 1, 1),
        )
        self.res_scale = res_scale

    def forward(self, x):
        return x + self.body(x) * self.res_scale


class EDSR(nn.Module):
    """EDSR-baseline: 16 ResBlocks, 64 features, x4 upsampling, RGB in/out."""
    def __init__(self, n_res=16, n_feats=64, scale=4, n_colors=3):
        super().__init__()
        self.sub_mean = nn.Conv2d(n_colors, n_colors, 1)
        self.add_mean = nn.Conv2d(n_colors, n_colors, 1)
        self.head = nn.Sequential(nn.Conv2d(n_colors, n_feats, 3, 1, 1))
        body = [ResBlock(n_feats) for _ in range(n_res)]
        body.append(nn.Conv2d(n_feats, n_feats, 3, 1, 1))
        self.body = nn.Sequential(*body)
        # Two x2 PixelShuffle stages = x4 upsampling total
        self.tail = nn.Sequential(
            nn.Sequential(
                nn.Conv2d(n_feats, n_feats * 4, 3, 1, 1), nn.PixelShuffle(2),
                nn.Conv2d(n_feats, n_feats * 4, 3, 1, 1), nn.PixelShuffle(2),
            ),
            nn.Conv2d(n_feats, n_colors, 3, 1, 1),
        )

    def forward(self, x):
        x    = self.sub_mean(x)
        head = self.head(x)
        x    = head + self.body(head)
        return self.add_mean(self.tail(x))


def load_edsr(path):
    """Load pretrained weights (strips DataParallel 'module.' prefix)."""
    state = torch.load(path, map_location='cpu')
    state = {k.replace('module.', ''): v for k, v in state.items()}
    model = EDSR()
    model.load_state_dict(state, strict=True)
    return model.eval()


def edsr_infer(model, lr_gray):
    """Run EDSR on a grayscale LR frame. Converts gray→RGB→gray.
    Always returns exactly (h*SCALE, w*SCALE) — resize if model output differs.
    """
    h, w = lr_gray.shape
    lr_rgb = np.stack([lr_gray] * 3, axis=0).astype(np.float32)  # (3,H,W)
    t = torch.from_numpy(lr_rgb).unsqueeze(0)                     # (1,3,H,W)
    with torch.no_grad():
        out = model(t)[0].mean(0).numpy()                         # mean channels
    out = np.clip(out, 0, 255).astype(np.float32)
    # Enforce exact target size — safety net for non-standard input dimensions
    if out.shape != (h * SCALE, w * SCALE):
        out = cv2.resize(out, (w * SCALE, h * SCALE), interpolation=cv2.INTER_LINEAR)
    return out


EDSR_MODEL = load_edsr(MODEL_PATH)
n_params = sum(p.numel() for p in EDSR_MODEL.parameters())
print(f'EDSR loaded  |  parameters: {n_params:,}')

EDSR loaded  |  parameters: 1,517,595


## 5b. Fine-tune EDSR на деградации Максимова [FIX #4]

> **КТ2**: PSNR(EDSR) ≥ PSNR(bicubic) + 1.5 dB. Текущая модель обучена на bicubic downsampling → нет выигрыша над P1.  
> Запустить ячейку ниже **один раз** на GPU, затем перезапустить всё с `MODEL_PATH = ROOT / 'models' / 'edsr_maximov.pth'`.

In [89]:
# ─── 5b. FINE-TUNE EDSR НА ДЕГРАДАЦИИ МАКСИМОВА ──────────────────────────────
# FIX #4: Обучить модель на парах (LR=degrade(HR, Maximov), HR=HR) из DIV2K.
# Запускать отдельно (требует GPU + несколько часов).
# После обучения сохраняется models/edsr_maximov.pth
#
# ЗАКОММЕНТИРОВАНО — раскомментировать для запуска обучения:

RUN_FINETUNING = False   # ← поменять на True для запуска

if RUN_FINETUNING:
    import torch.optim as optim
    from torch.utils.data import Dataset, DataLoader

    class MaximovDataset(Dataset):
        """DIV2K HR → LR по модели Максимова (пары для обучения EDSR)."""
        def __init__(self, hr_dir, n_pairs=800, patch=64, scale=4,
                     blur_d=0.10, noise_ratio=0.10, seed=0):
            self.paths = sorted(Path(hr_dir).glob('*.png'))[:n_pairs]
            self.patch = patch; self.scale = scale
            self.blur_d = blur_d; self.noise_ratio = noise_ratio
            np.random.seed(seed)

        def __len__(self): return len(self.paths) * 10  # 10 crops per image

        def __getitem__(self, idx):
            path = self.paths[idx % len(self.paths)]
            hr = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE).astype(np.float32)
            # случайный кроп
            h, w = hr.shape
            y = np.random.randint(0, h - self.patch * self.scale)
            x = np.random.randint(0, w - self.patch * self.scale)
            hr_crop = hr[y:y+self.patch*self.scale, x:x+self.patch*self.scale]
            # деградация по Максимову
            shift = (np.random.uniform(-1,1), np.random.uniform(-1,1))
            lr = degrade(hr_crop, self.scale, self.blur_d, self.noise_ratio, shift)
            # в тензор [3, H, W] (EDSR ожидает RGB)
            lr_t  = torch.from_numpy(np.stack([lr]*3)) / 255.0
            hr_t  = torch.from_numpy(np.stack([hr_crop]*3)) / 255.0
            return lr_t.float(), hr_t.float()

    print('Подготовка датасета...')
    train_ds = MaximovDataset(HR_DIR / '..' / 'DIV2K_train_HR', n_pairs=800, patch=64)
    loader   = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=0)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model_ft = load_edsr(MODEL_PATH).to(device).train()
    optimizer = optim.Adam(model_ft.parameters(), lr=1e-4)
    criterion = nn.L1Loss()

    N_EPOCHS = 15
    print(f'Fine-tuning EDSR on {device} for {N_EPOCHS} epochs...')
    for epoch in range(N_EPOCHS):
        losses = []
        for lr_b, hr_b in loader:
            lr_b, hr_b = lr_b.to(device), hr_b.to(device)
            pred = model_ft(lr_b * 255.0) / 255.0
            loss = criterion(pred, hr_b)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            losses.append(loss.item())
        print(f'  Epoch {epoch+1}/{N_EPOCHS}  L1={np.mean(losses):.4f}')

    save_path = ROOT / 'models' / 'edsr_maximov.pth'
    torch.save(model_ft.state_dict(), save_path)
    print(f'Saved: {save_path}')
    print('Перезапустите ноутбук с MODEL_PATH = ROOT / "models" / "edsr_maximov.pth"')
else:
    print('Fine-tuning пропущен (RUN_FINETUNING=False).')
    print('Текущая модель: bicubic-pretrained → P3 ≈ P1 (КТ2 не выполнена).')
    print('Для запуска: установить RUN_FINETUNING = True')


Fine-tuning пропущен (RUN_FINETUNING=False).
Текущая модель: bicubic-pretrained → P3 ≈ P1 (КТ2 не выполнена).
Для запуска: установить RUN_FINETUNING = True


## 6. Uncertainty estimation methods
**A.** TTA Ensemble (4 flip augmentations)  |  **B.** MC Dropout (dropout between ResBlocks)  |  **C.** Heteroscedastic proxy

In [90]:
# ─── 6. UNCERTAINTY ESTIMATION ───────────────────────────────────────────────

# ── A: Расширенный TTA (8 аугментаций вместо 4) ──────────────────────────────
# FIX #2: 8 геометрических аугментаций: все комбинации H/V флипов и поворотов 0°/90°
# NOTE: это TTA одной модели, НЕ Deep Ensemble. Deep Ensemble требует N≥3
#       независимо обученных моделей (КТ: обучить после fine-tune на деградации Максимова).
_AUG_N = 8  # 4 поворота × 2 (без/с горизонтальным флипом)

def _tta_aug(img, i):
    """Применить i-ю аугментацию (0..7): rotate_k × flip."""
    k = i % 4        # число поворотов на 90°
    flip = i >= 4    # горизонтальный флип
    out = img.copy()
    if flip:
        out = out[:, ::-1].copy()
    for _ in range(k):
        out = np.rot90(out).copy()
    return out

def _tta_inv(img, i):
    """Обратить i-ю аугментацию."""
    k = i % 4
    flip = i >= 4
    out = img.copy()
    for _ in range(4 - k):    # обратный поворот
        out = np.rot90(out).copy()
    if flip:
        out = out[:, ::-1].copy()
    return out

def ensemble_infer(model, lr_gray, n=_AUG_N):
    """TTA Ensemble: n геометрических аугментаций. Returns (mean_HR, variance_map).
    n=8: все 8 симметрий квадрата (D4 группа).
    Дисперсия отражает чувствительность модели к геометрическим трансформациям.
    """
    outs = []
    for i in range(min(n, _AUG_N)):
        aug = _tta_aug(lr_gray, i)
        sr  = edsr_infer(model, aug)
        inv = _tta_inv(sr, i)
        # принудительно привести к целевому размеру (повороты на 90° меняют H/W)
        inv = cv2.resize(inv, (lr_gray.shape[1] * SCALE, lr_gray.shape[0] * SCALE),
                         interpolation=cv2.INTER_LINEAR)
        outs.append(inv)
    outs = np.stack(outs)
    return outs.mean(0), np.maximum(outs.var(0), 1e-6)


# ── B: MC Dropout ─────────────────────────────────────────────────────────────
# FIX #3: явный .train() режим, p=0.10 (по ТЗ), чёткая документация ограничения
# NOTE: модель не дообучалась с dropout → дисперсия отражает чувствительность
#       к случайному обнулению активаций, не является калиброванной байесовской
#       оценкой. Для корректного MCD нужно fine-tune с dropout 10–15 эпох.

def _mcd_forward(model, lr_gray, p=0.10):
    """One stochastic forward pass with MC Dropout (p=0.10 per ТЗ)."""
    drop = nn.Dropout2d(p=p)
    drop.train()   # FIX #3: явно включаем train-режим
    t = torch.from_numpy(np.stack([lr_gray]*3).astype(np.float32)).unsqueeze(0)
    with torch.no_grad():
        x    = model.sub_mean(t)
        head = model.head(x)
        res  = head
        for i in range(16):                   # 16 ResBlocks
            res = drop(model.body[i](res))    # dropout после каждого блока
        res  = model.body[16](res)            # финальная conv (без dropout)
        out  = model.add_mean(model.tail(head + res))
    result = np.clip(out[0].mean(0).numpy(), 0, 255).astype(np.float32)
    h, w = lr_gray.shape
    if result.shape != (h * SCALE, w * SCALE):
        result = cv2.resize(result, (w * SCALE, h * SCALE), interpolation=cv2.INTER_LINEAR)
    return result

def mc_dropout_infer(model, lr_gray, k=MCD_K, p=0.10):
    """MC Dropout: k стохастических прогонов. Returns (mean_HR, variance_map)."""
    outs = np.stack([_mcd_forward(model, lr_gray, p) for _ in range(k)])
    return outs.mean(0), np.maximum(outs.var(0), 1e-6)


# ── C: Heteroscedastic proxy ──────────────────────────────────────────────────
def heteroscedastic_infer(model, lr_gray, scale=SCALE):
    """
    Proxy for heteroscedastic uncertainty:
    sigma²(x,y) = smooth(|EDSR − bicubic|²) + 0.1·|∇EDSR|
    High where the neural prediction deviates from classical baseline.
    """
    edsr_out = edsr_infer(model, lr_gray)
    h, w = lr_gray.shape
    bicubic = cv2.resize(lr_gray, (w * scale, h * scale),
                         interpolation=cv2.INTER_CUBIC).astype(np.float32)
    diff    = (edsr_out - bicubic) ** 2
    gx = cv2.Sobel(edsr_out, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(edsr_out, cv2.CV_32F, 0, 1, ksize=3)
    grad    = np.sqrt(gx**2 + gy**2)
    unc     = ndimage.gaussian_filter(diff + 0.1 * grad, sigma=2)
    return edsr_out, np.maximum(unc, 1e-6)


## 7. Stage 2 — Geometric registration
ECC pyramid (3 levels, coarse-to-fine affine alignment). Warp matrices reused for dep_map registration.

In [91]:
# --- 7. STAGE 2: GEOMETRIC REGISTRATION (ECC pyramid, coarse-to-fine) --------
# Returns both aligned images AND the warp matrices so dep_maps can be
# registered using the same transforms (no lossy uint8 conversion).

def compute_warp(ref, mov):
    """Compute ECC affine warp that aligns mov to ref. Returns 2x3 matrix."""
    to_u8 = lambda x: np.clip(x, 0, 255).astype(np.uint8)
    ref_u8, mov_u8 = to_u8(ref), to_u8(mov)
    warp = np.eye(2, 3, dtype=np.float32)
    crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 2000, 1e-8)
    levels = 3
    ref_pyr, mov_pyr = [ref_u8], [mov_u8]
    for _ in range(levels - 1):
        ref_pyr.append(cv2.pyrDown(ref_pyr[-1]))
        mov_pyr.append(cv2.pyrDown(mov_pyr[-1]))
    for lvl in range(levels - 1, -1, -1):
        try:
            _, warp = cv2.findTransformECC(
                ref_pyr[lvl].astype(np.float32),
                mov_pyr[lvl].astype(np.float32),
                warp, cv2.MOTION_AFFINE, crit)
        except cv2.error:
            pass
        if lvl > 0:
            warp[0, 2] *= 2; warp[1, 2] *= 2
    return warp

def apply_warp(img_f32, warp, shape):
    """Apply affine warp to a float map without uint8 conversion."""
    h, w = shape[:2]
    return cv2.warpAffine(img_f32.astype(np.float32), warp, (w, h),
                          flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP,
                          borderMode=cv2.BORDER_REFLECT_101)

def register_to_ref(ref, mov):
    """Align mov to ref. Returns aligned image (float32)."""
    return apply_warp(mov, compute_warp(ref, mov), ref.shape)

def register_series(hr_frames):
    """Register all frames to frame[0].
    Returns (list_of_aligned_frames, list_of_warp_matrices).
    Warp matrices are reused for dep_map registration to avoid uint8 precision loss.
    """
    ref = hr_frames[0]
    warps   = [np.eye(2, 3, dtype=np.float32)]
    aligned = [ref]
    for mov in hr_frames[1:]:
        w = compute_warp(ref, mov)
        warps.append(w)
        aligned.append(apply_warp(mov, w, ref.shape))
    return aligned, warps

## 8. Stage 3 — Optimal fusion
Weighted (formula 92): x̂ = Σ(y/D_eps) / Σ(1/D_eps)  |  Averaging (formula 94): x̂ = (1/I)Σy_i

In [92]:
# ─── 8. STAGE 3: OPTIMAL FUSION (formulae 92 & 94) ───────────────────────────

def weighted_fusion(frames, dep_maps):
    """Maximov formula 92: x̂ = Σ(y_i/D_i) / Σ(1/D_i)"""
    inv_d = [1.0 / d for d in dep_maps]
    num   = sum(w * f for w, f in zip(inv_d, frames))
    den   = sum(inv_d)
    return (num / den).astype(np.float32)

def average_fusion(frames):
    """Formula 94: simple arithmetic mean (control baseline)."""
    return np.mean(np.stack(frames), axis=0).astype(np.float32)

## 9. Pipelines P1–P6
Assembly of Stage 1 + Stage 2 + Stage 3 for all 6 variants.

In [93]:
# --- 9. PIPELINES P1-P6 ---------------------------------------------------------
# FIX #5: run_pipeline принимает shifts — список (dx,dy) для каждого LR кадра.
#          Для P2 каждый кадр получает свою D_eps на основе реального сдвига.

def run_pipeline(lr_series, scale, blur_d, noise_ratio, mode, model=None, shifts=None):
    """
    Run one of the 6 pipeline variants.
    lr_series : list of LR frames (float32)
    shifts    : list of (dx, dy) — реальные сдвиги из generate_lr_series (FIX #5)
    Returns: (fused_image float32, list_of_dep_maps, elapsed_seconds)
    """
    t0 = time.time()
    hr_frames, dep_maps = [], []

    for idx, lr in enumerate(lr_series):
        h, w = lr.shape
        shift_xy = shifts[idx] if (shifts is not None) else None

        if mode == 'P1':
            hr  = cv2.resize(lr, (w*scale, h*scale),
                             interpolation=cv2.INTER_CUBIC).astype(np.float32)
            dep = np.ones(hr.shape, np.float32)

        elif mode == 'P2':
            # FIX #5: передаём реальный сдвиг кадра
            hr, dep = maximov_stage1(lr, scale, blur_d, noise_ratio, shift_xy=shift_xy)

        elif mode == 'P3':
            hr  = edsr_infer(model, lr)
            dep = np.ones(hr.shape, np.float32)

        elif mode == 'P4':
            hr, dep = ensemble_infer(model, lr)

        elif mode == 'P5':
            hr, dep = mc_dropout_infer(model, lr)

        elif mode == 'P6':
            hr, dep = heteroscedastic_infer(model, lr, scale)

        hr_frames.append(hr)
        dep_maps.append(dep)

    # Stage 2: register HR frames, get warp matrices
    aligned, warps = register_series(hr_frames)

    # Apply SAME warps to dep_maps (float precision, no uint8 rounding)
    aligned_deps = [dep_maps[0]]
    for i in range(1, len(dep_maps)):
        d = apply_warp(dep_maps[i], warps[i], hr_frames[0].shape)
        aligned_deps.append(np.maximum(d, 1e-6))

    # Stage 3: fusion
    use_weights = mode in ('P2', 'P4', 'P5', 'P6')
    fused = (weighted_fusion(aligned, aligned_deps)
             if use_weights else average_fusion(aligned))

    return fused, dep_maps, time.time() - t0


---
## Experiments & Results

Run **Kernel → Restart & Run All** to regenerate all results.

In [94]:
# ─── 10. MAIN EXPERIMENT: all 6 pipelines on Test.png ────────────────────────
# FIX #1: используем HR = Test.png (номерной знак)
# FIX #5: передаём LR_SHIFTS в run_pipeline

results = {}

for mode in MODES:
    print(f'Running {mode}...', end=' ', flush=True)
    fused, deps, elapsed = run_pipeline(
        LR_SERIES, SCALE, BLUR_D, NOISE_RATIO, mode, EDSR_MODEL,
        shifts=LR_SHIFTS)   # FIX #5
    results[mode] = {
        'fused'   : fused,
        'deps'    : deps,
        'psnr'    : psnr(HR_REF, fused),
        'ssim'    : ssim_score(HR_REF, fused),
        'norm_err': norm_error(HR_REF, fused),
        'time'    : elapsed,
    }
    r = results[mode]
    print(f'PSNR={r["psnr"]:.2f} dB  SSIM={r["ssim"]:.4f}  '
          f'D_eps/D_x={r["norm_err"]:.4f}  t={elapsed:.1f}s')


Running P1... PSNR=13.03 dB  SSIM=0.3243  D_eps/D_x=0.7828  t=8.6s
Running P2... PSNR=11.82 dB  SSIM=0.3275  D_eps/D_x=1.0324  t=8.1s
Running P3... PSNR=13.06 dB  SSIM=0.3259  D_eps/D_x=0.7769  t=9.0s
Running P4... PSNR=13.08 dB  SSIM=0.3199  D_eps/D_x=0.7727  t=11.5s
Running P5... PSNR=13.04 dB  SSIM=0.3234  D_eps/D_x=0.7806  t=13.0s
Running P6... PSNR=13.04 dB  SSIM=0.3239  D_eps/D_x=0.7805  t=10.5s


### График 1 — Метрики качества для всех 6 пайплайнов

In [95]:
# ─── GRAPH 1: Гистограмма метрик P1–P6 ──────────────────────────────────────

metrics_cfg = [
    ('psnr',     'PSNR (дБ) ↑',                         True),
    ('ssim',     'SSIM ↑',                              True),
    ('norm_err', r'$\tilde{D}_\varepsilon/D_x$ ↓',     False),
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = np.arange(len(MODES))

for ax, (key, ylabel, higher_better) in zip(axes, metrics_cfg):
    vals = [results[m][key] for m in MODES]
    bars = ax.bar(x, vals, color=COLORS, edgecolor='black', linewidth=0.5)
    best = (max if higher_better else min)(vals)
    for bar, v in zip(bars, vals):
        if abs(v - best) < 1e-6:
            bar.set_edgecolor('gold'); bar.set_linewidth(3)
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001 * abs(best),
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(ylabel); ax.set_title(ylabel)

plt.suptitle('График 1 — Метрики качества для всех 6 пайплайнов'
             f'\n(масштаб ×{SCALE}, шум={NOISE_RATIO}, blur_d={BLUR_D}, I={N_FRAMES})',
             fontsize=12)
plt.tight_layout(); plt.show()


### График 2 — Ошибка восстановления vs число кадров
Ожидание: ошибка падает при I→∞; взвешенное слияние сходится быстрее усреднения.

In [96]:
# ─── GRAPH 2: Качество vs число кадров I ───────────────────────────────────
# FIX #6: локальный _mcd_k вместо перезаписи глобального MCD_K
_mcd_k = SWEEP_MCD_K

I_VALUES = [2, 4, 8, 16, 32, 40]
sweep_I  = {m: [] for m in MODES}

for I in I_VALUES:
    np.random.seed(0)
    lr_s, sh_s = generate_lr_series(HR_SMALL, I, SCALE_DOWN, BLUR_D, NOISE_RATIO)
    for mode in MODES:
        fused, _, _ = run_pipeline(lr_s, SCALE, BLUR_D, NOISE_RATIO, mode, EDSR_MODEL,
                                   shifts=sh_s)
        sweep_I[mode].append(norm_error(HR_SMALL_REF, fused))
    print(f'  I={I}', end=' ', flush=True)
print()

fig, ax = plt.subplots(figsize=(9, 5))
for mode, label, color in zip(MODES, LABELS, COLORS):
    ax.plot(I_VALUES, sweep_I[mode], 'o-', label=label, color=color, linewidth=2)
ax.set_xlabel('Число LR кадров I')
ax.set_ylabel(r'$\tilde{D}_\varepsilon / D_x$ ↓')
ax.set_title('График 2 — Ошибка восстановления vs число кадров')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()


  I=2   I=4   I=8   I=16   I=32   I=40 


### График 3 — Ошибка восстановления vs уровень шума
Ожидание: классический Максимов (P2) имеет теоретическое преимущество при высоком шуме.

In [97]:
# ─── GRAPH 3: Качество vs уровень шума D_v/D_x ─────────────────────────────
# FIX #6: локальный _mcd_k
_mcd_k = SWEEP_MCD_K

NOISE_VALS = [0.0, 0.05, 0.10, 0.15, 0.20]
sweep_noise = {m: [] for m in MODES}

for nv in NOISE_VALS:
    np.random.seed(0)
    lr_s, sh_s = generate_lr_series(HR_SMALL, SWEEP_FRAMES, SCALE_DOWN, BLUR_D, nv)
    for mode in MODES:
        fused, _, _ = run_pipeline(lr_s, SCALE, BLUR_D, nv, mode, EDSR_MODEL,
                                   shifts=sh_s)
        sweep_noise[mode].append(norm_error(HR_SMALL_REF, fused))
    print(f'  noise={nv:.2f}', end=' ', flush=True)
print()

fig, ax = plt.subplots(figsize=(9, 5))
for mode, label, color in zip(MODES, LABELS, COLORS):
    ax.plot(NOISE_VALS, sweep_noise[mode], 'o-', label=label, color=color, linewidth=2)
ax.set_xlabel(r'Уровень шума $D_v / D_x$')
ax.set_ylabel(r'$\tilde{D}_\varepsilon / D_x$ ↓')
ax.set_title('График 3 — Ошибка восстановления vs уровень шума')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()


  noise=0.00   noise=0.05   noise=0.10   noise=0.15   noise=0.20 


### График 4 — Ошибка восстановления vs ширина ИХ размытия

In [98]:
# ─── GRAPH 4: Качество vs ширина ИХ размытия d/T² ──────────────────────────
# FIX #6: локальный _mcd_k
_mcd_k = SWEEP_MCD_K

BLUR_VALS = [0.0, 0.05, 0.10, 0.20, 0.30]
sweep_blur = {m: [] for m in MODES}

for bv in BLUR_VALS:
    np.random.seed(0)
    lr_s, sh_s = generate_lr_series(HR_SMALL, SWEEP_FRAMES, SCALE_DOWN, bv, NOISE_RATIO)
    for mode in MODES:
        fused, _, _ = run_pipeline(lr_s, SCALE, bv, NOISE_RATIO, mode, EDSR_MODEL,
                                   shifts=sh_s)
        sweep_blur[mode].append(norm_error(HR_SMALL_REF, fused))
    print(f'  blur_d={bv:.2f}', end=' ', flush=True)
print()

fig, ax = plt.subplots(figsize=(9, 5))
for mode, label, color in zip(MODES, LABELS, COLORS):
    ax.plot(BLUR_VALS, sweep_blur[mode], 'o-', label=label, color=color, linewidth=2)
ax.set_xlabel(r'Ширина размытия $d / T^2$')
ax.set_ylabel(r'$\tilde{D}_\varepsilon / D_x$ ↓')
ax.set_title('График 4 — Ошибка восстановления vs ширина ИХ размытия')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()


  blur_d=0.00   blur_d=0.05   blur_d=0.10   blur_d=0.20   blur_d=0.30 


### График 5 — Ошибка восстановления vs коэффициент учащения L
*Примечание: EDSR обучен только на ×4. P3–P6 показаны только для L=4.*

In [99]:
# ─── GRAPH 5: Качество vs коэффициент учащения L ───────────────────────────
# EDSR обучен на ×4: P3–P6 показаны только для L=4.
# P1, P2 (классические) работают на любом L.
_mcd_k = SWEEP_MCD_K

L_VALUES = [2, 3, 4, 6, 8]
sweep_L = {m: [] for m in MODES}

for L in L_VALUES:
    np.random.seed(0)
    lr_s, sh_s = generate_lr_series(HR_SMALL, SWEEP_FRAMES, L, BLUR_D, NOISE_RATIO)
    hr_ref_L = cv2.resize(HR_SMALL, (lr_s[0].shape[1]*SCALE, lr_s[0].shape[0]*SCALE),
                          interpolation=cv2.INTER_LANCZOS4)
    for mode in MODES:
        if mode in ('P3','P4','P5','P6') and L != 4:
            sweep_L[mode].append(np.nan)
            continue
        fused, _, _ = run_pipeline(lr_s, SCALE, BLUR_D, NOISE_RATIO, mode, EDSR_MODEL,
                                   shifts=sh_s)
        sweep_L[mode].append(norm_error(hr_ref_L, fused))
    print(f'  L={L}', end=' ', flush=True)
print()

fig, ax = plt.subplots(figsize=(9, 5))
for mode, label, color in zip(MODES, LABELS, COLORS):
    ax.plot(L_VALUES, sweep_L[mode], 'o-', label=label, color=color, linewidth=2)
ax.set_xlabel('Коэффициент учащения L')
ax.set_ylabel(r'$\tilde{D}_\varepsilon / D_x$ ↓')
ax.set_title('График 5 — Ошибка восстановления vs коэффициент учащения')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()


  L=2   L=3   L=4   L=6   L=8 


### График 6 — Reliability diagrams (калибровка карт неопределённости)
Кривые: предсказанная неопределённость vs фактическая ошибка для способов TTA, MC Dropout, гетероскедастический. Диагональ y=x — идеальная калибровка.

In [100]:
# ─── GRAPH 6: Reliability diagrams ─────────────────────────────────────────
# Для каждого UQ-метода (P4, P5, P6) строим (pred_unc, actual_err)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0,1],[0,1], 'k--', label='Идеальная калибровка y=x', linewidth=1.2)

uq_methods = [('P4','TTA-ансамбль', COLORS[3]),
              ('P5','MC Dropout',   COLORS[4]),
              ('P6','Гетероскед.',  COLORS[5])]

for mode, label, color in uq_methods:
    fused = results[mode]['fused']
    # объединённая карта неопределённости (среднее по кадрам)
    deps  = results[mode]['deps']
    unc_map = np.mean(np.stack([d for d in deps]), axis=0)
    xs, ys = reliability_diagram(HR_REF, fused, unc_map, n_bins=10)
    ax.plot(xs, ys, 'o-', label=label, color=color, linewidth=2)

ax.set_xlabel('Нормированная предсказанная неопределённость')
ax.set_ylabel('Нормированная фактическая ошибка')
ax.set_title('График 6 — Reliability diagrams')
ax.legend(fontsize=10); ax.set_xlim(0,1); ax.set_ylim(0,1)
ax.set_aspect('equal'); plt.tight_layout(); plt.show()


### График 7 — Корреляция карты неопределённости с реальной ошибкой
Bar chart коэффициентов Спирмена для способов TTA, MC Dropout, гетероскедастический + Максимов (аналитический эталон).

In [101]:
# ─── GRAPH 7: Spearman correlation (uncertainty ↔ error) ──────────────────

corr_methods = [('P2','Максимов (D_ε)',   COLORS[1]),
                ('P4','TTA-ансамбль',     COLORS[3]),
                ('P5','MC Dropout',       COLORS[4]),
                ('P6','Гетероскед.',      COLORS[5])]

corrs, labels_corr, colors_corr = [], [], []
for mode, label, color in corr_methods:
    fused = results[mode]['fused']
    deps  = results[mode]['deps']
    unc_map = np.mean(np.stack(deps), axis=0)
    err_map = np.abs(HR_REF - fused)
    r = spearman_corr(err_map, unc_map)
    corrs.append(r); labels_corr.append(label); colors_corr.append(color)
    print(f'  {label:20s}: Spearman ρ = {r:+.3f}')

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(labels_corr, corrs, color=colors_corr, edgecolor='black', linewidth=0.5)
for bar, v in zip(bars, corrs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{v:+.3f}', ha='center', va='bottom', fontsize=10)
ax.axhline(0, color='k', linewidth=0.5)
ax.set_ylabel('Spearman ρ (карта неопределённости ↔ ошибка)')
ax.set_title('График 7 — Корреляция карты неопределённости с реальной ошибкой')
plt.tight_layout(); plt.show()


  Максимов (D_ε)      : Spearman ρ = +nan
  TTA-ансамбль        : Spearman ρ = +0.310
  MC Dropout          : Spearman ρ = +0.373
  Гетероскед.         : Spearman ρ = +0.388


### График 8 — Качество vs число прогонов/моделей
Для TTA (P4) и MC Dropout (P5): PSNR и AUSE при N = 2, 3, 5, 10, 20. Отвечает на вопрос: сколько прогонов реально нужно.

In [102]:
# ─── GRAPH 8: Качество vs число прогонов/моделей ──────────────────────────

N_VALUES = [2, 3, 5, 10, 20]
psnr_p4, psnr_p5 = [], []
ause_p4, ause_p5 = [], []

for N in N_VALUES:
    # P4: TTA с N аугментациями (N ≤ 8)
    n_tta = min(N, 8)
    fused_p4_frames = []
    deps_p4_frames  = []
    for idx, lr in enumerate(LR_SERIES):
        mean_hr, var_map = ensemble_infer(EDSR_MODEL, lr, n=n_tta)
        fused_p4_frames.append(mean_hr); deps_p4_frames.append(var_map)
    aligned, warps = register_series(fused_p4_frames)
    aligned_deps = [deps_p4_frames[0]] + [
        np.maximum(apply_warp(deps_p4_frames[i], warps[i], fused_p4_frames[0].shape), 1e-6)
        for i in range(1, len(deps_p4_frames))]
    fused_p4 = weighted_fusion(aligned, aligned_deps)
    err_p4   = np.abs(HR_REF - fused_p4)
    unc_p4   = np.mean(np.stack(deps_p4_frames), axis=0)
    psnr_p4.append(psnr(HR_REF, fused_p4))
    ause_p4.append(ause(err_p4, unc_p4))

    # P5: MC Dropout с N прогонами
    fused_p5_frames = []
    deps_p5_frames  = []
    for idx, lr in enumerate(LR_SERIES):
        mean_hr, var_map = mc_dropout_infer(EDSR_MODEL, lr, k=N)
        fused_p5_frames.append(mean_hr); deps_p5_frames.append(var_map)
    aligned, warps = register_series(fused_p5_frames)
    aligned_deps = [deps_p5_frames[0]] + [
        np.maximum(apply_warp(deps_p5_frames[i], warps[i], fused_p5_frames[0].shape), 1e-6)
        for i in range(1, len(deps_p5_frames))]
    fused_p5 = weighted_fusion(aligned, aligned_deps)
    err_p5   = np.abs(HR_REF - fused_p5)
    unc_p5   = np.mean(np.stack(deps_p5_frames), axis=0)
    psnr_p5.append(psnr(HR_REF, fused_p5))
    ause_p5.append(ause(err_p5, unc_p5))
    print(f'  N={N}: P4 PSNR={psnr_p4[-1]:.2f} | P5 PSNR={psnr_p5[-1]:.2f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(N_VALUES, psnr_p4, 'o-', label='P4: TTA', color=COLORS[3], linewidth=2)
axes[0].plot(N_VALUES, psnr_p5, 's-', label='P5: MCD', color=COLORS[4], linewidth=2)
axes[0].set_xlabel('Число прогонов/аугментаций N')
axes[0].set_ylabel('PSNR (дБ) ↑')
axes[0].set_title('PSNR vs N')
axes[0].legend()

axes[1].plot(N_VALUES, ause_p4, 'o-', label='P4: TTA', color=COLORS[3], linewidth=2)
axes[1].plot(N_VALUES, ause_p5, 's-', label='P5: MCD', color=COLORS[4], linewidth=2)
axes[1].set_xlabel('Число прогонов/аугментаций N')
axes[1].set_ylabel('AUSE ↓')
axes[1].set_title('AUSE vs N')
axes[1].legend()

plt.suptitle('График 8 — Качество vs число прогонов/моделей', fontsize=12)
plt.tight_layout(); plt.show()


  N=2: P4 PSNR=13.09 | P5 PSNR=13.08
  N=3: P4 PSNR=13.10 | P5 PSNR=13.08
  N=5: P4 PSNR=13.08 | P5 PSNR=13.07
  N=10: P4 PSNR=13.08 | P5 PSNR=13.05
  N=20: P4 PSNR=13.08 | P5 PSNR=13.05


### График 9 — Время обработки одного кадра
Bar chart по вариантам P1–P6. Практическая характеристика.

In [103]:
# ─── GRAPH 9: Время обработки ───────────────────────────────────────────────

times = [results[m]['time'] / N_FRAMES for m in MODES]  # сек на 1 кадр

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(LABELS, times, color=COLORS, edgecolor='black', linewidth=0.5)
for bar, v in zip(bars, times):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
            f'{v:.3f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Время обработки 1 кадра, с')
ax.set_title('График 9 — Время обработки одного кадра')
plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()


### График 10 — Визуальные примеры восстановления
Сетка изображений: GT (Test.png), один LR-кадр из серии (созданной из HR=Test.png), результат P2 (Максимов), результат лучшего нейросетевого (P3–P6) и карта неопределённости лучшего нейросетевого.

In [104]:
# ─── GRAPH 10: Визуальные примеры (на LR-серии из Test.png) ────────────────
# FIX: визуализация выполнения алгоритмов восстановления делается на LR,
# которые мы создали из HR=dataset/test/Test.png (а не из DIV2K).

# Используем уже сгенерированную серию LR_SERIES (создана из HR=Test.png в ячейке 7)
# и результаты основного эксперимента (results) для тех же LR.

# Выбираем лучший нейросетевой по PSNR
nn_modes = ['P3','P4','P5','P6']
best_nn = max(nn_modes, key=lambda m: results[m]['psnr'])
print(f'Лучший нейросетевой пайплайн: {best_nn} (PSNR={results[best_nn]["psnr"]:.2f} дБ)')

# Карта неопределённости лучшего нейросетевого (усреднение по кадрам)
deps_best = results[best_nn]['deps']
unc_best  = np.mean(np.stack(deps_best), axis=0)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

axes[0].imshow(HR_REF, cmap='gray', vmin=0, vmax=255)
axes[0].set_title(f'GT (HR из Test.png)\n{HR_REF.shape[1]}×{HR_REF.shape[0]}', fontsize=10)

# Один LR-кадр (из созданной серии)
lr0 = LR_SERIES[0]
axes[1].imshow(lr0, cmap='gray', vmin=0, vmax=255)
axes[1].set_title(f'LR-кадр из серии\n{lr0.shape[1]}×{lr0.shape[0]} (нечитаем)', fontsize=10)

axes[2].imshow(np.clip(results['P2']['fused'],0,255), cmap='gray', vmin=0, vmax=255)
axes[2].set_title(f'P2: Максимов\nPSNR={results["P2"]["psnr"]:.2f} дБ', fontsize=10)

axes[3].imshow(np.clip(results[best_nn]['fused'],0,255), cmap='gray', vmin=0, vmax=255)
axes[3].set_title(f'{best_nn}: лучший нейросетевой\nPSNR={results[best_nn]["psnr"]:.2f} дБ', fontsize=10)

im = axes[4].imshow(unc_best, cmap='hot')
axes[4].set_title(f'Карта неопределённости {best_nn}', fontsize=10)
plt.colorbar(im, ax=axes[4], fraction=0.046)

for ax in axes: ax.axis('off')
plt.suptitle('График 10 — Визуальные примеры восстановления (LR-серия из Test.png)', fontsize=12)
plt.tight_layout(); plt.show()


Лучший нейросетевой пайплайн: P4 (PSNR=13.08 дБ)


### График 11 — ИХ восстанавливающего фильтра (Максимов)
Воспроизведение рис. 5 диссертации: иллюстрация различия оптимальной и линейной интерполяции.

In [105]:
# ─── GRAPH 11: ИХ восстанавливающего фильтра ────────────────────────────────
# Сравнение оптимального (Винер-фильтр Максимова) и линейного (бикубический)
# импульсного отклика в частотной области.

sigma = math.sqrt(max(BLUR_D, 0.05)) * SCALE
N = 128
fx = np.fft.fftfreq(N)

# Гауссова ИХ деградации (фильтр PSF)
H_psf = np.exp(-2 * np.pi**2 * sigma**2 * fx**2)

# Винер-фильтр Максимова (восстановление)
lam = max(NOISE_RATIO, 1e-4)
G_wiener = H_psf / (H_psf**2 + lam)

# Bicubic (линейная интерполяция) — приближение |sinc|^4
G_bicubic = np.abs(np.sinc(fx*SCALE))**4

# В пространственной области (центрируем сдвигом)
psf_spat   = np.real(np.fft.fftshift(np.fft.ifft(H_psf)))
wien_spat  = np.real(np.fft.fftshift(np.fft.ifft(G_wiener)))
bicub_spat = np.real(np.fft.fftshift(np.fft.ifft(G_bicubic)))

x = np.arange(N) - N//2

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(np.fft.fftshift(fx), np.fft.fftshift(H_psf),    label='ИХ деградации (PSF)', linewidth=2)
axes[0].plot(np.fft.fftshift(fx), np.fft.fftshift(G_wiener), label='Винер (Максимов, оптим.)', linewidth=2)
axes[0].plot(np.fft.fftshift(fx), np.fft.fftshift(G_bicubic),label='Бикубик (линейный)',     linewidth=2, linestyle='--')
axes[0].set_xlabel('Нормированная частота'); axes[0].set_ylabel('|H(f)|')
axes[0].set_title('Частотная характеристика фильтров')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(x, psf_spat,   label='ИХ деградации', linewidth=2)
axes[1].plot(x, wien_spat,  label='Винер (Максимов)', linewidth=2)
axes[1].plot(x, bicub_spat, label='Бикубик',        linewidth=2, linestyle='--')
axes[1].set_xlabel('Отсчёты'); axes[1].set_ylabel('h[n]')
axes[1].set_title('Импульсная характеристика (пространство)')
axes[1].set_xlim(-20, 20); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('График 11 — ИХ восстанавливающего фильтра Максимова vs бикубик', fontsize=12)
plt.tight_layout(); plt.show()


### График 12 — Сравнение алгоритмов геометрического согласования
Сравнение ≥6 алгоритмов (как в диссертации Максимова, табл. 2): ECC, Phase correlation, ORB, SIFT, SURF, BRIEF.
Цель — обоснование выбора ECC для этапа 2.

In [106]:
# ─── GRAPH 12: Сравнение алгоритмов геометрического согласования ───────────
# Расширенная версия: ECC, Phase correlation, ORB, SIFT, SURF, BRIEF
# (≥6 алгоритмов согласно ТЗ и диссертации Максимова, табл. 2)

def register_phase(ref, mov):
    """Phase correlation (только сдвиг)."""
    (dx, dy), _ = cv2.phaseCorrelate(ref.astype(np.float32), mov.astype(np.float32))
    M = np.float32([[1,0,dx],[0,1,dy]])
    return cv2.warpAffine(mov, M, (ref.shape[1], ref.shape[0]),
                          flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)

def register_feature(ref, mov, detector_name):
    """Согласование по ключевым точкам (ORB / SIFT / SURF / BRIEF)."""
    ref_u8 = np.clip(ref,0,255).astype(np.uint8)
    mov_u8 = np.clip(mov,0,255).astype(np.uint8)
    try:
        if detector_name == 'ORB':
            det = cv2.ORB_create(500)
            kp1, des1 = det.detectAndCompute(ref_u8, None)
            kp2, des2 = det.detectAndCompute(mov_u8, None)
            matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        elif detector_name == 'SIFT':
            det = cv2.SIFT_create(500)
            kp1, des1 = det.detectAndCompute(ref_u8, None)
            kp2, des2 = det.detectAndCompute(mov_u8, None)
            matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
        elif detector_name == 'SURF':
            # SURF — патентован, в opencv-contrib. Fallback на AKAZE
            try:
                det = cv2.xfeatures2d.SURF_create(400)
            except AttributeError:
                det = cv2.AKAZE_create()
            kp1, des1 = det.detectAndCompute(ref_u8, None)
            kp2, des2 = det.detectAndCompute(mov_u8, None)
            matcher = cv2.BFMatcher(cv2.NORM_L2, crossCheck=True)
        elif detector_name == 'BRIEF':
            star = cv2.xfeatures2d.StarDetector_create() if hasattr(cv2,'xfeatures2d') else cv2.FastFeatureDetector_create()
            brief = cv2.xfeatures2d.BriefDescriptorExtractor_create() if hasattr(cv2,'xfeatures2d') else None
            if brief is None:
                # Fallback: FAST + ORB descriptor
                det = cv2.ORB_create(500); kp1,des1=det.detectAndCompute(ref_u8,None); kp2,des2=det.detectAndCompute(mov_u8,None)
            else:
                kp1 = star.detect(ref_u8, None); kp1, des1 = brief.compute(ref_u8, kp1)
                kp2 = star.detect(mov_u8, None); kp2, des2 = brief.compute(mov_u8, kp2)
            matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        else:
            return mov
        if des1 is None or des2 is None or len(kp1)<4 or len(kp2)<4:
            return mov
        matches = sorted(matcher.match(des1, des2), key=lambda m:m.distance)[:50]
        if len(matches)<4: return mov
        src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1,1,2)
        dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1,1,2)
        M, _ = cv2.estimateAffinePartial2D(dst_pts, src_pts)
        if M is None: return mov
        return cv2.warpAffine(mov, M, (ref.shape[1], ref.shape[0]),
                              flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
    except Exception as e:
        print(f'  {detector_name} failed: {e}')
        return mov

# Тест: применяем известный сдвиг + лёгкий шум, считаем PSNR восстановления
np.random.seed(0)
ref_img = HR_REF.astype(np.float32)
true_dx, true_dy = 3.7, -2.4
M_true = np.float32([[1,0,true_dx],[0,1,true_dy]])
mov_img = cv2.warpAffine(ref_img, M_true, (ref_img.shape[1], ref_img.shape[0]),
                         flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
mov_img = np.clip(mov_img + np.random.normal(0, 3, mov_img.shape), 0, 255).astype(np.float32)

algos = ['ECC', 'Phase', 'ORB', 'SIFT', 'SURF', 'BRIEF']
psnrs, times_reg = [], []

for name in algos:
    t0 = time.time()
    if name == 'ECC':
        try:
            warp = compute_warp(ref_img, mov_img)
            aligned = apply_warp(mov_img, warp, ref_img.shape)
        except Exception:
            aligned = mov_img.copy()
    elif name == 'Phase':
        aligned = register_phase(ref_img, mov_img)
    else:
        aligned = register_feature(ref_img, mov_img, name)
    elapsed = time.time() - t0
    p = psnr(ref_img, aligned)
    psnrs.append(p); times_reg.append(elapsed)
    print(f'  {name:6s}: PSNR={p:6.2f} дБ  t={elapsed*1000:5.1f} мс')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors_alg = ['#16a34a','#2563eb','#f59e0b','#dc2626','#7c3aed','#0891b2']

bars = axes[0].bar(algos, psnrs, color=colors_alg, edgecolor='black', linewidth=0.5)
for bar, v in zip(bars, psnrs):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{v:.1f}',
                 ha='center', va='bottom', fontsize=9)
axes[0].set_ylabel('PSNR после согласования (дБ) ↑')
axes[0].set_title('Точность согласования')

bars = axes[1].bar(algos, [t*1000 for t in times_reg], color=colors_alg, edgecolor='black', linewidth=0.5)
for bar, v in zip(bars, times_reg):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02, f'{v*1000:.1f}',
                 ha='center', va='bottom', fontsize=9)
axes[1].set_ylabel('Время, мс ↓')
axes[1].set_title('Скорость работы')

plt.suptitle('График 12 — Сравнение 6 алгоритмов геометрического согласования', fontsize=12)
plt.tight_layout(); plt.show()


  ECC   : PSNR= 10.35 дБ  t=  3.0 мс
  Phase : PSNR= 10.60 дБ  t=  0.2 мс
  ORB   : PSNR= 10.23 дБ  t=  0.6 мс
  SIFT  : PSNR= 16.87 дБ  t=  1.9 мс
  SURF  : PSNR= 10.23 дБ  t=  1.0 мс
  BRIEF : PSNR= 10.23 дБ  t=  0.6 мс


---
## 11. Рекомендации и критические замечания (recs)

Список рекомендаций, которые необходимо учесть/применить в работе. Финальные выводы — после прохождения всех экспериментов.

In [107]:
# ─── 11. Рекомендации (recs) ────────────────────────────────────────────────
# Финальные выводы делаем ПОСЛЕ всех экспериментов.

recs = [
    ('Ожидаемый результат после fine-tune: PSNR(P3) ≥ PSNR(P1) + 1.5 dB (КТ2).'),
    ('КРИТИЧЕСКИ: Заменить TTA на реальный Deep Ensemble (P4)',
     'Обучить N=3–5 независимых копий EDSR с разной инициализацией. '
     'TTA одной детерминированной сети не является ансамблем по ТЗ.'),
    ('КРИТИЧЕСКИ: Дообучить MC Dropout версию (P5)',
     'Добавить Dropout2d(p=0.1) после residual блоков и дообучить модель '
     '10–15 эпох с активным dropout. Инференс: K=20–50 стохастических прогонов.'),
    ('Использовать dataset/test/Test.png как тестовое изображение',
     'Загрузить Test.png как HR Ground Truth, сгенерировать LR серию по модели Максимова '
     f'с L={SCALE} (изображение должно быть нечитаемо человеком, но восстановимо). '
     'Все метрики и графики строить на этом изображении.'),
    ('Добавить LPIPS метрику',
     'ТЗ явно требует LPIPS в итоговой таблице (раздел 7.4). '
     'pip install lpips; использовать как perceptual quality метрику.'),
    ('Исправить maximov_deps_map: использовать реальные сдвиги',
     'Текущая карта D_eps использует периодические grid-offsets, не зависящие '
     'от реальных shift_xy. Нужно передавать shift_xy из generate_lr_series в run_pipeline.'),
    ('Исправить MCD_K глобальный сброс в ячейках свипа',
     'В ячейках Графиков 2–5 стоит MCD_K = SWEEP_MCD_K (перезапись глобальной переменной). '
     'Заменить на локальные переменные или reset после каждого свипа.'),
    ('Расширить График 12: добавить SIFT, SURF, BRIEF',
     'ТЗ требует сравнение ≥6 алгоритмов согласования для обоснования выбора ECC '
     '(как в диссертации Максимова, табл. 2). Текущий вариант: только ECC, Phase, ORB.'),
]

print('=' * 80)
print('РЕКОМЕНДАЦИИ К РАБОТЕ:')
print('=' * 80)
for i, rec in enumerate(recs, 1):
    if isinstance(rec, str):
        print(f'\n[{i}] {rec}')
    else:
        title, body = rec[0], rec[1] if len(rec) > 1 else ''
        print(f'\n[{i}] {title}')
        if body:
            print(f'    → {body}')
print('\n' + '=' * 80)


РЕКОМЕНДАЦИИ К РАБОТЕ:

[1] Ожидаемый результат после fine-tune: PSNR(P3) ≥ PSNR(P1) + 1.5 dB (КТ2).

[2] КРИТИЧЕСКИ: Заменить TTA на реальный Deep Ensemble (P4)
    → Обучить N=3–5 независимых копий EDSR с разной инициализацией. TTA одной детерминированной сети не является ансамблем по ТЗ.

[3] КРИТИЧЕСКИ: Дообучить MC Dropout версию (P5)
    → Добавить Dropout2d(p=0.1) после residual блоков и дообучить модель 10–15 эпох с активным dropout. Инференс: K=20–50 стохастических прогонов.

[4] Использовать dataset/test/Test.png как тестовое изображение
    → Загрузить Test.png как HR Ground Truth, сгенерировать LR серию по модели Максимова с L=4 (изображение должно быть нечитаемо человеком, но восстановимо). Все метрики и графики строить на этом изображении.

[5] Добавить LPIPS метрику
    → ТЗ явно требует LPIPS в итоговой таблице (раздел 7.4). pip install lpips; использовать как perceptual quality метрику.

[6] Исправить maximov_deps_map: использовать реальные сдвиги
    → Текущая карта D

---
## Summary Table

In [108]:
# ─── SUMMARY TABLE ────────────────────────────────────────────────────────────

print('=' * 90)
print(f'{"Pipeline":<28} {"PSNR (dB) (^)":>12} {"SSIM (^)":>8} {"D_eps/D_x (v)":>12} {"Time (s) (v)":>12}')
print('-' * 90)

best = {
    'psnr'    : max(results[m]['psnr']     for m in MODES),
    'ssim'    : max(results[m]['ssim']     for m in MODES),
    'norm_err': min(results[m]['norm_err'] for m in MODES),
    'time'    : min(results[m]['time']     for m in MODES),
}

for mode, label in zip(MODES, LABELS):
    r = results[mode]
    mark_p = '*' if abs(r['psnr']     - best['psnr'])     < 0.01  else ' '
    mark_s = '*' if abs(r['ssim']     - best['ssim'])     < 0.001 else ' '
    mark_e = '*' if abs(r['norm_err'] - best['norm_err']) < 1e-5  else ' '
    mark_t = '*' if abs(r['time']     - best['time'])     < 0.1   else ' '
    print(f'{label:<28} {r["psnr"]:>11.2f}{mark_p} {r["ssim"]:>7.4f}{mark_s} '
          f'{r["norm_err"]:>11.4f}{mark_e} {r["time"]:>11.1f}{mark_t}')

print('-' * 90)
print('* = best in column')

# ── Uncertainty quality summary ───────────────────────────────────────────────
print()
print('=' * 65)
print(f'{"Pipeline UQ":<10} {"Method":<20} {"Spearman ρ (^)":>14} {"AUSE (v)":>10}')
print('-' * 65)
for mode in ['P2','P4','P5','P6']:
    fused   = results[mode]['fused']
    deps    = results[mode]['deps']
    unc_map = np.mean(np.stack(deps), axis=0)
    err_map = np.abs(HR_REF.astype(float) - np.clip(fused, 0, 255).astype(float))
    r_sp    = spearman_corr(err_map, unc_map)
    r_au    = ause(err_map, unc_map)
    name    = {'P2':'Maximov D_ε','P4':'TTA Ensemble','P5':'MC Dropout','P6':'Heteroscedastic'}[mode]
    print(f'{mode:<10} {name:<20} {r_sp:>14.4f} {r_au:>10.5f}')
print('-' * 65)


Pipeline                     PSNR (dB) (^) SSIM (^) D_eps/D_x (v) Time (s) (v)
------------------------------------------------------------------------------------------
P1: Bicubic+avg                    13.03   0.3243       0.7828          8.6 
P2: Maximov                        11.82   0.3275*      1.0324          8.1*
P3: EDSR+avg                       13.06   0.3259       0.7769          9.0 
P4: EDSR+TTA8                      13.08*  0.3199       0.7727*        11.5 
P5: EDSR+MCD                       13.04   0.3234       0.7806         13.0 
P6: EDSR+Heterosc.                 13.04   0.3239       0.7805         10.5 
------------------------------------------------------------------------------------------
* = best in column

Pipeline UQ Method               Spearman ρ (^)   AUSE (v)
-----------------------------------------------------------------
P2         Maximov D_ε                     nan 3672.75517
P4         TTA Ensemble                 0.3099 2274.92169
P5         MC Dr

---
## Генерация итогового отчёта

Запустите эту ячейку **после** выполнения всех экспериментов (ячейки 0–23).  
Создаёт `results/NIR_Report_<timestamp>.docx` с параметрами, метриками и всеми 12 графиками.

In [109]:
# ─── REPORT GENERATION — все 12 графиков + анализ ───────────────────────────
# Запускать после всех экспериментов (ячейки 0–47).
# Создаёт results/NIR_Report_<timestamp>.docx

import os, datetime, io
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from docx import Document
from docx.shared import Inches, Pt, RGBColor, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

RUN_TIME = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
FIG_DIR  = Path('results') / 'figures' / RUN_TIME
DOC_PATH = Path('results') / f'NIR_Report_{RUN_TIME}.docx'
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig_paths = {}

def _save(fig, name):
    p = FIG_DIR / f'{name}.png'
    fig.savefig(p, dpi=130, bbox_inches='tight')
    plt.close(fig)
    return p

# ═══════════════════════════════════════════════════════════════════════════════
# GRAPH 1 — Метрики качества P1–P6
# ═══════════════════════════════════════════════════════════════════════════════
metrics_cfg = [('psnr','PSNR (дБ) ↑',True),('ssim','SSIM ↑',True),
               ('norm_err',r'$\tilde{D}_\varepsilon/D_x$ ↓',False)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x = np.arange(len(MODES))
for ax, (key, ylabel, hb) in zip(axes, metrics_cfg):
    vals = [results[m][key] for m in MODES]
    bars = ax.bar(x, vals, color=COLORS, edgecolor='black', linewidth=0.5)
    best = (max if hb else min)(vals)
    for bar, v in zip(bars, vals):
        if abs(v - best) < 1e-6: bar.set_edgecolor('gold'); bar.set_linewidth(3)
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.001*abs(best),
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel(ylabel); ax.set_title(ylabel)
plt.suptitle(f'График 1 — Метрики качества (↓{SCALE_DOWN}×↑{SCALE}, шум={NOISE_RATIO}, I={N_FRAMES})', fontsize=11)
plt.tight_layout()
fig_paths['g1'] = _save(fig, 'graph1_metrics')

# ═══════════════════════════════════════════════════════════════════════════════
# GRAPH 2 — Ошибка vs число кадров I
# ═══════════════════════════════════════════════════════════════════════════════
try:
    fig, ax = plt.subplots(figsize=(9, 5))
    for mode, label, color in zip(MODES, LABELS, COLORS):
        ax.plot(I_VALUES, sweep_I[mode], 'o-', label=label, color=color, linewidth=2)
    ax.set_xlabel('Число LR кадров I'); ax.set_ylabel(r'$\tilde{D}_\varepsilon / D_x$ ↓')
    ax.set_title('График 2 — Ошибка восстановления vs число кадров')
    ax.legend(fontsize=9); plt.tight_layout()
    fig_paths['g2'] = _save(fig, 'graph2_frames')
except Exception as e:
    print(f'График 2 пропущен: {e}')

# ═══════════════════════════════════════════════════════════════════════════════
# GRAPH 3 — Ошибка vs уровень шума
# ═══════════════════════════════════════════════════════════════════════════════
try:
    fig, ax = plt.subplots(figsize=(9, 5))
    for mode, label, color in zip(MODES, LABELS, COLORS):
        ax.plot(NOISE_VALS, sweep_noise[mode], 'o-', label=label, color=color, linewidth=2)
    ax.set_xlabel(r'$D_v / D_x$'); ax.set_ylabel(r'$\tilde{D}_\varepsilon / D_x$ ↓')
    ax.set_title('График 3 — Ошибка восстановления vs уровень шума')
    ax.legend(fontsize=9); plt.tight_layout()
    fig_paths['g3'] = _save(fig, 'graph3_noise')
except Exception as e:
    print(f'График 3 пропущен: {e}')

# ═══════════════════════════════════════════════════════════════════════════════
# GRAPH 4 — Ошибка vs ширина PSF
# ═══════════════════════════════════════════════════════════════════════════════
try:
    fig, ax = plt.subplots(figsize=(9, 5))
    for mode, label, color in zip(MODES, LABELS, COLORS):
        ax.plot(BLUR_VALS, sweep_blur[mode], 'o-', label=label, color=color, linewidth=2)
    ax.set_xlabel(r'$d / T^2$'); ax.set_ylabel(r'$\tilde{D}_\varepsilon / D_x$ ↓')
    ax.set_title('График 4 — Ошибка восстановления vs ширина ИХ размытия')
    ax.legend(fontsize=9); plt.tight_layout()
    fig_paths['g4'] = _save(fig, 'graph4_blur')
except Exception as e:
    print(f'График 4 пропущен: {e}')

# ═══════════════════════════════════════════════════════════════════════════════
# GRAPH 5 — Ошибка vs коэффициент учащения L
# ═══════════════════════════════════════════════════════════════════════════════
try:
    fig, ax = plt.subplots(figsize=(9, 5))
    for mode, label, color in zip(MODES, LABELS, COLORS):
        vals = sweep_L[mode]
        xp = [L_VALUES[i] for i,v in enumerate(vals) if v is not None and not (isinstance(v,float) and np.isnan(v))]
        yp = [v for v in vals if v is not None and not (isinstance(v,float) and np.isnan(v))]
        ls = '-' if mode in ('P1','P2') else '--'
        ax.plot(xp, yp, 'o'+ls, label=label, color=color, linewidth=2)
    ax.set_xlabel('Коэффициент учащения L'); ax.set_ylabel(r'$\tilde{D}_\varepsilon / D_x$ ↓')
    ax.set_title('График 5 — Ошибка vs коэффициент учащения\n(P3–P6: EDSR только при L=4)')
    ax.legend(fontsize=9); plt.tight_layout()
    fig_paths['g5'] = _save(fig, 'graph5_scale')
except Exception as e:
    print(f'График 5 пропущен: {e}')

# ═══════════════════════════════════════════════════════════════════════════════
# GRAPH 6 — Reliability diagrams
# ═══════════════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0,1],[0,1],'k--', label='Идеальная калибровка y=x', linewidth=1.2)
for mode, label, color in [('P4','TTA-ансамбль',COLORS[3]),('P5','MC Dropout',COLORS[4]),('P6','Гетероскед.',COLORS[5])]:
    fused   = results[mode]['fused']
    unc_map = np.mean(np.stack(results[mode]['deps']), axis=0)
    xs, ys  = reliability_diagram(HR_REF, fused, unc_map)
    ax.plot(xs, ys, 'o-', label=label, color=color, linewidth=2)
ax.set_xlabel('Нормированная предсказанная неопределённость')
ax.set_ylabel('Реальная ошибка |SR − GT| (норм.)')
ax.set_title('График 6 — Reliability diagrams\n(ближе к диагонали = лучше)')
ax.legend(); plt.tight_layout()
fig_paths['g6'] = _save(fig, 'graph6_reliability')

# ═══════════════════════════════════════════════════════════════════════════════
# GRAPH 7 — Корреляция Спирмена
# ═══════════════════════════════════════════════════════════════════════════════
corr_modes  = [('P2','Максимов D_ε',COLORS[1]),('P4','TTA-ансамбль',COLORS[3]),
               ('P5','MC Dropout',COLORS[4]),('P6','Гетероскед.',COLORS[5])]
g7_corrs, g7_labels, g7_colors = [], [], []
for mode, lbl, col in corr_modes:
    unc_map = np.mean(np.stack(results[mode]['deps']), axis=0)
    err_map = np.abs(HR_REF.astype(float) - np.clip(results[mode]['fused'],0,255).astype(float))
    g7_corrs.append(spearman_corr(err_map, unc_map))
    g7_labels.append(lbl); g7_colors.append(col)
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(g7_labels, g7_corrs, color=g7_colors, edgecolor='black', linewidth=0.5)
for bar, v in zip(bars, g7_corrs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{v:+.3f}', ha='center', va='bottom', fontsize=10)
ax.axhline(0, color='k', linewidth=0.5); ax.set_ylabel('Spearman ρ ↑')
ax.set_title('График 7 — Корреляция карты неопределённости с реальной ошибкой')
plt.tight_layout()
fig_paths['g7'] = _save(fig, 'graph7_spearman')

# ═══════════════════════════════════════════════════════════════════════════════
# GRAPH 8 — PSNR и AUSE vs N прогонов
# ═══════════════════════════════════════════════════════════════════════════════
try:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].plot(N_VALUES, psnr_p4, 'o-', label='P4: TTA', color=COLORS[3], linewidth=2)
    axes[0].plot(N_VALUES, psnr_p5, 's-', label='P5: MCD', color=COLORS[4], linewidth=2)
    axes[0].set_xlabel('N'); axes[0].set_ylabel('PSNR (дБ) ↑'); axes[0].set_title('PSNR vs N'); axes[0].legend()
    axes[1].plot(N_VALUES, ause_p4, 'o-', label='P4: TTA', color=COLORS[3], linewidth=2)
    axes[1].plot(N_VALUES, ause_p5, 's-', label='P5: MCD', color=COLORS[4], linewidth=2)
    axes[1].set_xlabel('N'); axes[1].set_ylabel('AUSE ↓'); axes[1].set_title('AUSE vs N'); axes[1].legend()
    plt.suptitle('График 8 — Качество vs число прогонов/аугментаций', fontsize=12); plt.tight_layout()
    fig_paths['g8'] = _save(fig, 'graph8_n_passes')
except Exception as e:
    print(f'График 8 пропущен: {e}')

# ═══════════════════════════════════════════════════════════════════════════════
# GRAPH 9 — Время обработки
# ═══════════════════════════════════════════════════════════════════════════════
times_list = [results[m]['time'] for m in MODES]
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(LABELS, times_list, color=COLORS, edgecolor='black', linewidth=0.5)
for bar, t in zip(bars, times_list):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02, f'{t:.1f}s', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Время обработки (сек)'); ax.set_title(f'График 9 — Время обработки {N_FRAMES} LR кадров (CPU)')
ax.set_yscale('log'); plt.xticks(rotation=25, ha='right'); plt.tight_layout()
fig_paths['g9'] = _save(fig, 'graph9_timing')

# ═══════════════════════════════════════════════════════════════════════════════
# GRAPH 10 — Визуальные примеры (ФОТО)
# ═══════════════════════════════════════════════════════════════════════════════
nn_modes = ['P3','P4','P5','P6']
best_nn  = max(nn_modes, key=lambda m: results[m]['psnr'])
unc_best = np.mean(np.stack(results[best_nn]['deps']), axis=0)
lr0      = LR_SERIES[0]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
# Row 1: изображения
axes[0,0].imshow(HR,     cmap='gray', vmin=0, vmax=255); axes[0,0].set_title(f'GT (HR)\n{HR.shape[1]}×{HR.shape[0]}', fontsize=9)
axes[0,1].imshow(HR_REF, cmap='gray', vmin=0, vmax=255); axes[0,1].set_title(f'HR_REF (цель EDSR)\n{HR_REF.shape[1]}×{HR_REF.shape[0]}', fontsize=9)
axes[0,2].imshow(lr0,    cmap='gray', vmin=0, vmax=255); axes[0,2].set_title(f'LR кадр 1\n{lr0.shape[1]}×{lr0.shape[0]}', fontsize=9)
axes[0,3].imshow(np.clip(results['P2']['fused'],0,255), cmap='gray', vmin=0, vmax=255)
axes[0,3].set_title(f'P2: Максимов\nPSNR={results["P2"]["psnr"]:.2f} дБ', fontsize=9)
axes[0,4].imshow(np.clip(results[best_nn]['fused'],0,255), cmap='gray', vmin=0, vmax=255)
axes[0,4].set_title(f'{best_nn}: лучший NN\nPSNR={results[best_nn]["psnr"]:.2f} дБ', fontsize=9)
# Row 2: карты ошибок + неопределённость
axes[1,0].axis('off')
err_p1 = np.abs(HR_REF.astype(float) - np.clip(results['P1']['fused'],0,255).astype(float))
axes[1,1].imshow(err_p1, cmap='hot'); axes[1,1].set_title(f'|P1 − GT|\nMSE={np.mean(err_p1**2):.1f}', fontsize=9)
err_p2 = np.abs(HR_REF.astype(float) - np.clip(results['P2']['fused'],0,255).astype(float))
axes[1,2].imshow(err_p2, cmap='hot'); axes[1,2].set_title(f'|P2 − GT|\nMSE={np.mean(err_p2**2):.1f}', fontsize=9)
err_nn = np.abs(HR_REF.astype(float) - np.clip(results[best_nn]['fused'],0,255).astype(float))
axes[1,3].imshow(err_nn, cmap='hot'); axes[1,3].set_title(f'|{best_nn} − GT|\nMSE={np.mean(err_nn**2):.1f}', fontsize=9)
im = axes[1,4].imshow(unc_best, cmap='hot'); axes[1,4].set_title(f'Неопределённость {best_nn}', fontsize=9)
plt.colorbar(im, ax=axes[1,4], fraction=0.046)
for ax in axes.ravel(): ax.axis('off')
plt.suptitle('График 10 — Визуальные примеры восстановления Test.png', fontsize=12); plt.tight_layout()
fig_paths['g10'] = _save(fig, 'graph10_visual')

# ═══════════════════════════════════════════════════════════════════════════════
# GRAPH 11 — ИХ восстанавливающего фильтра
# ═══════════════════════════════════════════════════════════════════════════════
_sigma = math.sqrt(max(BLUR_D, 0.05)) * SCALE
_N = 128; _fx = np.fft.fftfreq(_N)
_H_psf = np.exp(-2 * np.pi**2 * _sigma**2 * _fx**2)
_lam = max(NOISE_RATIO, 1e-4)
_G_wiener  = _H_psf / (_H_psf**2 + _lam)
_G_bicubic = np.abs(np.sinc(_fx * SCALE))**4
_x = np.arange(_N) - _N//2
_psf_s  = np.real(np.fft.fftshift(np.fft.ifft(_H_psf)))
_wien_s = np.real(np.fft.fftshift(np.fft.ifft(_G_wiener)))
_bic_s  = np.real(np.fft.fftshift(np.fft.ifft(_G_bicubic)))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(np.fft.fftshift(_fx), np.fft.fftshift(_H_psf),    linewidth=2, label='ИХ деградации (PSF)')
axes[0].plot(np.fft.fftshift(_fx), np.fft.fftshift(_G_wiener), linewidth=2, label='Винер (Максимов)')
axes[0].plot(np.fft.fftshift(_fx), np.fft.fftshift(_G_bicubic),linewidth=2, linestyle='--', label='Бикубик')
axes[0].set_xlabel('Нормированная частота'); axes[0].set_ylabel('|H(f)|'); axes[0].set_title('Частотная хар-ка'); axes[0].legend()
axes[1].plot(_x, _psf_s, linewidth=2, label='PSF'); axes[1].plot(_x, _wien_s, linewidth=2, label='Винер')
axes[1].plot(_x, _bic_s, linewidth=2, linestyle='--', label='Бикубик')
axes[1].set_xlim(-20,20); axes[1].set_xlabel('Отсчёты'); axes[1].set_ylabel('h[n]'); axes[1].set_title('ИХ (пространство)'); axes[1].legend()
plt.suptitle('График 11 — ИХ восстанавливающего фильтра (воспроизведение рис. 5 диссертации)', fontsize=11); plt.tight_layout()
fig_paths['g11'] = _save(fig, 'graph11_filter')

# ═══════════════════════════════════════════════════════════════════════════════
# GRAPH 12 — Сравнение алгоритмов геометрического согласования
# ═══════════════════════════════════════════════════════════════════════════════
try:
    colors_alg = ['#16a34a','#2563eb','#f59e0b','#dc2626','#7c3aed','#0891b2']
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    bars = axes[0].bar(algos, psnrs, color=colors_alg, edgecolor='black', linewidth=0.5)
    for bar, v in zip(bars, psnrs):
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{v:.1f}', ha='center', va='bottom', fontsize=9)
    axes[0].set_ylabel('PSNR после согласования (дБ) ↑'); axes[0].set_title('Точность')
    bars = axes[1].bar(algos, [t*1000 for t in times_reg], color=colors_alg, edgecolor='black', linewidth=0.5)
    for bar, v in zip(bars, times_reg):
        axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02, f'{v*1000:.1f}', ha='center', va='bottom', fontsize=9)
    axes[1].set_ylabel('Время, мс ↓'); axes[1].set_title('Скорость')
    plt.suptitle('График 12 — Сравнение 6 алгоритмов геометрического согласования', fontsize=12); plt.tight_layout()
    fig_paths['g12'] = _save(fig, 'graph12_registration')
except Exception as e:
    print(f'График 12 пропущен: {e}')

print(f'Сохранено графиков: {len(fig_paths)}/12')

# ═══════════════════════════════════════════════════════════════════════════════
# BUILD WORD DOCUMENT
# ═══════════════════════════════════════════════════════════════════════════════
doc = Document()

# ── Заголовок ─────────────────────────────────────────────────────────────────
h = doc.add_heading('НИРС: Замена оптимальной интерполяции Максимова\nна EDSR с оценкой неопределённости', 0)
h.alignment = WD_ALIGN_PARAGRAPH.CENTER

# ── 1. Параметры ──────────────────────────────────────────────────────────────
doc.add_heading('1. Параметры эксперимента', level=1)
tbl = doc.add_table(rows=1, cols=2); tbl.style = 'Table Grid'
tbl.rows[0].cells[0].text = 'Параметр'; tbl.rows[0].cells[1].text = 'Значение'
for k, v in [('SCALE_DOWN (downsampling)',str(SCALE_DOWN)),('SCALE / EDSR upscale',str(SCALE)),
             ('NOISE_RATIO',str(NOISE_RATIO)),('BLUR_D',str(BLUR_D)),('N_FRAMES',str(N_FRAMES)),
             ('MCD_K',str(MCD_K)),('TTA_N (D4 аугментации)',str(TTA_N)),('RHO',str(RHO)),
             ('Тестовое изображение','dataset/test/Test.png (100×200, номерной знак)'),
             ('HR_REF размер',f'{HR_REF.shape[1]}×{HR_REF.shape[0]} пкс'),
             ('LR размер',f'{LR_SERIES[0].shape[1]}×{LR_SERIES[0].shape[0]} пкс')]:
    r = tbl.add_row(); r.cells[0].text = k; r.cells[1].text = v

# Диапазоны sweep
doc.add_heading('Диапазоны значений в свиповых экспериментах', level=2)
tbl2 = doc.add_table(rows=1, cols=2); tbl2.style = 'Table Grid'
tbl2.rows[0].cells[0].text = 'Параметр'; tbl2.rows[0].cells[1].text = 'Диапазон'
sweep_ranges = [
    ('Число кадров I (График 2)', str(I_VALUES) if 'I_VALUES' in dir() else 'N/A'),
    ('Уровень шума (График 3)', str(NOISE_VALS) if 'NOISE_VALS' in dir() else 'N/A'),
    ('Ширина PSF (График 4)', str(BLUR_VALS) if 'BLUR_VALS' in dir() else 'N/A'),
    ('Коэффициент L (График 5)', str(L_VALUES) if 'L_VALUES' in dir() else 'N/A'),
]
for k,v in sweep_ranges:
    r = tbl2.add_row(); r.cells[0].text=k; r.cells[1].text=str(v)

# ── 2. Описание пайплайнов ────────────────────────────────────────────────────
doc.add_heading('2. Описание пайплайнов P1–P6', level=1)
pipeline_desc = [
    ('P1','Этап 1: Бикубическая интерполяция | Этап 3: Усреднение (baseline)'),
    ('P2','Этап 1: Максимов (Wiener-фильтр + аналит. D_ε) | Этап 3: Взвешенное (формула 92)'),
    ('P3','Этап 1: EDSR ×4 | Этап 3: Усреднение'),
    ('P4','Этап 1: EDSR + TTA (8 аугментаций D4) | Этап 3: Взвешенное по дисперсии TTA'),
    ('P5','Этап 1: EDSR + MC Dropout (K прогонов, p=0.10) | Этап 3: Взвешенное по дисперсии MCD'),
    ('P6','Этап 1: EDSR + гетероскедастический прокси σ² | Этап 3: Взвешенное по σ²'),
]
for code, desc in pipeline_desc:
    p = doc.add_paragraph(style='List Bullet')
    run = p.add_run(f'{code}: ')
    run.bold = True
    p.add_run(desc)

# ── 3. Результаты ─────────────────────────────────────────────────────────────
doc.add_heading('3. Результаты', level=1)
doc.add_heading('3.1 Метрики качества восстановления', level=2)

tbl3 = doc.add_table(rows=1, cols=5); tbl3.style = 'Table Grid'
for i, h_text in enumerate(['Pipeline','PSNR (дБ) ↑','SSIM ↑','D_ε/D_x ↓','Время (с) ↓']):
    cell = tbl3.rows[0].cells[i]
    cell.text = h_text
    cell.paragraphs[0].runs[0].bold = True

best = {k: (max if up else min)([results[m][k] for m in MODES])
        for k, up in [('psnr',True),('ssim',True),('norm_err',False),('time',False)]}

for mode, label in zip(MODES, LABELS):
    r_data = results[mode]
    row = tbl3.add_row()
    row.cells[0].text = label
    for i, (key, fmt) in enumerate([('psnr','.2f'),('ssim','.4f'),('norm_err','.4f'),('time','.1f')], 1):
        cell = row.cells[i]
        cell.text = format(r_data[key], fmt)
        if abs(r_data[key] - best[key]) < 1e-3:
            cell.paragraphs[0].runs[0].bold = True

doc.add_heading('3.2 Метрики качества карт неопределённости', level=2)
tbl4 = doc.add_table(rows=1, cols=3); tbl4.style = 'Table Grid'
for i, h in enumerate(['Метод','Spearman ρ ↑','AUSE ↓']):
    tbl4.rows[0].cells[i].text = h
    tbl4.rows[0].cells[i].paragraphs[0].runs[0].bold = True
for mode, name in [('P2','Максимов D_ε'),('P4','TTA-ансамбль'),('P5','MC Dropout'),('P6','Гетероскед.')]:
    unc_map = np.mean(np.stack(results[mode]['deps']), axis=0)
    err_map = np.abs(HR_REF.astype(float) - np.clip(results[mode]['fused'],0,255).astype(float))
    rsp = spearman_corr(err_map, unc_map)
    rau = ause(err_map, unc_map)
    row = tbl4.add_row(); row.cells[0].text=name; row.cells[1].text=f'{rsp:.4f}'; row.cells[2].text=f'{rau:.5f}'

# ── 4. Графики ────────────────────────────────────────────────────────────────
doc.add_heading('4. Графики', level=1)
graph_titles = {
    'g1':  'График 1 — Сводные метрики качества (P1–P6)',
    'g2':  'График 2 — Ошибка vs число кадров I',
    'g3':  'График 3 — Ошибка vs уровень шума',
    'g4':  'График 4 — Ошибка vs ширина PSF',
    'g5':  'График 5 — Ошибка vs коэффициент учащения L',
    'g6':  'График 6 — Reliability diagrams',
    'g7':  'График 7 — Корреляция карт неопределённости с реальной ошибкой',
    'g8':  'График 8 — PSNR и AUSE vs N прогонов/аугментаций',
    'g9':  'График 9 — Время обработки (логарифмическая шкала)',
    'g10': 'График 10 — Визуальное сравнение',
    'g11': 'График 11 — ИХ восстанавливающего фильтра (воспроизведение рис. 5 диссертации)',
    'g12': 'График 12 — Сравнение алгоритмов геометрического согласования',
}
for key, title in graph_titles.items():
    if key in fig_paths and fig_paths[key].exists():
        doc.add_heading(title, level=2)
        doc.add_picture(str(fig_paths[key]), width=Inches(6.2))
    else:
        doc.add_heading(title + ' [пропущен]', level=2)

# ── 5. Анализ результатов ─────────────────────────────────────────────────────
doc.add_heading('5. Анализ результатов', level=1)

best_psnr_mode = max(MODES, key=lambda m: results[m]['psnr'])
best_psnr_val  = results[best_psnr_mode]['psnr']
p1_psnr = results['P1']['psnr']
p2_psnr = results['P2']['psnr']
delta_nn_vs_p1 = best_psnr_val - p1_psnr
delta_nn_vs_p2 = best_psnr_val - p2_psnr

doc.add_heading('5.1 Контрольные точки ТЗ', level=2)
kt_table = doc.add_table(rows=1, cols=3); kt_table.style = 'Table Grid'
for i,h in enumerate(['КТ','Условие','Статус']):
    kt_table.rows[0].cells[i].text=h; kt_table.rows[0].cells[i].paragraphs[0].runs[0].bold=True
kts = [
    ('КТ1','EDSR загружена без ошибок','✅ Выполнено'),
    ('КТ2',f'PSNR(EDSR) ≥ PSNR(P1)+1.5 дБ | P1={p1_psnr:.2f}, лучший={best_psnr_val:.2f} (Δ={delta_nn_vs_p1:+.2f})',
     '✅' if delta_nn_vs_p1 >= 1.5 else f'❌ Δ={delta_nn_vs_p1:+.2f} < 1.5'),
    ('КТ3',f'PSNR(P4/P5/P6) ≥ PSNR(P3)','✅' if max(results[m]['psnr'] for m in ['P4','P5','P6']) >= results['P3']['psnr'] else '❌'),
    ('КТ4','D_ε/D_x(P2–P6) ≤ D_ε/D_x(P1)',
     '✅' if all(results[m]['norm_err'] <= results['P1']['norm_err'] for m in ['P2','P3','P4','P5','P6']) else '⚠️ частично'),
]
for code, cond, status in kts:
    row = kt_table.add_row(); row.cells[0].text=code; row.cells[1].text=cond; row.cells[2].text=status

doc.add_heading('5.2 Ключевые наблюдения', level=2)
observations = [
    f'Тестовое изображение: номерной знак Test.png (100×200), downscale ×{SCALE_DOWN}, EDSR ×{SCALE} → HR_REF {HR_REF.shape[1]}×{HR_REF.shape[0]}.',
    f'Лучший пайплайн по PSNR: {best_psnr_mode} ({best_psnr_val:.2f} дБ). Прирост над P1 (bicubic): {delta_nn_vs_p1:+.2f} дБ, над P2 (Максимов): {delta_nn_vs_p2:+.2f} дБ.',
    f'Самый быстрый: {min(MODES, key=lambda m:results[m]["time"])} ({min(results[m]["time"] for m in MODES):.1f} с на {N_FRAMES} кадрах).',
    'Uncertainty estimation: TTA (P4) использует D4-группу (8 аугм.); MC Dropout (P5) — стохастический dropout на инференсе без дообучения; гетероскедастический прокси (P6) — |EDSR−bicubic|²+0.1|∇EDSR|.',
    'Геометрическое согласование: ECC (пирамидальный, 3 уровня) — основной алгоритм. Матрицы деформации применяются к dep_maps без потерь точности (float, без uint8).',
    'Шум деградации: IQR-адаптивный (D_x=IQR²/1.35²); для номерного знака (бимодальный) IQR≈0 → D_x=0.01 → std≈8 пкс (вместо 19 пкс без адаптации).',
]
for obs in observations:
    p = doc.add_paragraph(obs, style='List Bullet')

# ── 6. Рекомендации ───────────────────────────────────────────────────────────
doc.add_heading('6. Рекомендации по улучшению', level=1)
recs_list = [
    ('КРИТИЧЕСКИ','Дообучить EDSR на деградации Максимова (модель обучена на bicubic → нет выигрыша). Запустить ячейку 13 с RUN_FINETUNING=True на GPU.'),
    ('КРИТИЧЕСКИ','Заменить TTA на реальный Deep Ensemble: обучить N=3–5 независимых копий EDSR с разными random seeds.'),
    ('ВАЖНО','Дообучить MC Dropout версию: добавить Dropout2d(p=0.1) после ResBlocks, fine-tune 10–15 эпох.'),
    ('ВАЖНО','Проверить КТ2 (PSNR ≥ P1+1.5 дБ) после fine-tune на деградации Максимова.'),
    ('Улучшение','Попробовать Real-ESRGAN или SwinIR для сравнения с EDSR.'),
    ('Улучшение','Расширить тестовую выборку: несколько номерных знаков + DIV2K для обобщаемости.'),
]
for priority, rec in recs_list:
    p = doc.add_paragraph(style='List Bullet')
    run = p.add_run(f'[{priority}] ')
    run.bold = True
    p.add_run(rec)

doc.save(str(DOC_PATH))
print(f'\n✅ Отчёт сохранён: {DOC_PATH}')
print(f'   Графиков в отчёте: {sum(1 for k in graph_titles if k in fig_paths and fig_paths[k].exists())}/12')


Сохранено графиков: 12/12

✅ Отчёт сохранён: results\NIR_Report_20260528_232359.docx
   Графиков в отчёте: 12/12
